# Forecast Engine — Walkthrough

End-to-end demonstration of the hybrid forecasting engine: profile a series, backtest the model zoo, select with calibration and cost awareness, forecast with honest intervals, and detect drift.

**Principle:** forecasts come from forecasting models; LLMs reason about the system.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from forecast_engine import ForecastEngine, ForecastSpec
from forecast_engine.types import OutputForm, LossAsymmetry
from forecast_engine.monitoring import DriftMonitor
from forecast_engine.iteration import AnomalyDetector

## 1. A synthetic series with trend + seasonality

In [ ]:
np.random.seed(42)
n = 730
dates = pd.date_range('2023-01-01', periods=n, freq='D')
series = pd.Series(
    np.linspace(100,200,n) + 10*np.sin(2*np.pi*np.arange(n)/7)
    + 20*np.sin(2*np.pi*np.arange(n)/365) + np.random.randn(n)*5,
    index=dates, name='mau')
series.plot(figsize=(12,4), title='Synthetic MAU series'); plt.show()

## 2. Define the decision, not the model

Here under-forecasting is 3x costlier (capacity-style decision).

In [ ]:
spec = ForecastSpec(
    name='mau', horizon=28, granularity='D',
    output_form=OutputForm.DISTRIBUTION,
    loss_asymmetry=LossAsymmetry.UNDER_COSTLY, asymmetry_ratio=3.0,
    series_value='high')
engine = ForecastEngine(spec, n_backtest_windows=4)
selection = engine.fit_select(series)
print(selection.strategy, '|', selection.model, '|', selection.rationale)

## 3. The honest backtest leaderboard

In [ ]:
engine.leaderboard()

## 4. Forecast with calibrated intervals

In [ ]:
fc = engine.forecast(series, horizon=28)
ax = series.iloc[-90:].plot(figsize=(12,5), label='history')
fc_df = fc.to_frame()
ax.plot(fc_df.index, fc_df['point'], label='forecast', color='C1')
ax.fill_between(fc_df.index, fc_df['lower_80'], fc_df['upper_80'], alpha=0.3, color='C1', label='80%')
ax.fill_between(fc_df.index, fc_df['lower_95'], fc_df['upper_95'], alpha=0.15, color='C1', label='95%')
ax.legend(); plt.title('Forecast with calibrated intervals'); plt.show()

## 5. Drift monitoring

In [ ]:
monitor = DriftMonitor()
ref = series.iloc[:365].values
recent = series.iloc[365:].values
data_drift = monitor.detect_data_drift(ref, recent)
print('PSI:', round(data_drift.psi,3), '| detected:', data_drift.detected, '| severity:', data_drift.severity)

## Next steps

- Add the heavy model families (`pip install statsforecast lightgbm neuralforecast chronos-forecasting`) to populate the full zoo
- Wire the `llmops` layer to your LLM provider for framing, drift narration, and executive narratives
- See `examples/quickstart.py` for the scriptable version